In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json

In [ ]:
BASE_URL = "https://teatarkomedija.mk"

response = requests.get(BASE_URL)
response.status_code

In [ ]:
html = response.text
html[112330:112412]

In [ ]:
soup = BeautifulSoup(html, "html.parser")
soup.title

In [ ]:
all_cards = soup.find_all("div")
len(all_cards)
# all_cards[200].prettify()

In [ ]:
detail_links = []

for show in all_cards:
    a = show.select_one("div.qodef-e-media-image > a")
    if a and a.get("href"):

        detail_links.append(a["href"])

detail_links

In [ ]:
len(detail_links)

In [ ]:
df_detail_links = pd.DataFrame(data=detail_links, columns=["href"])

df_detail_links

In [ ]:
unique_detail_links = df_detail_links["href"].unique()

unique_detail_links

In [ ]:
len(unique_detail_links)

In [ ]:
def clean_text(text):
    if not text:
        return None

    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")

    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
def get_theater_address_info(soup):

    footer_widget = soup.select_one("div.textwidget")
    if not footer_widget:
        return {"location": "Unknown location!", "city": "Unknown City!"}

    raw_text = footer_widget.get_text(separator=" ", strip=True)
    # НУ Театар Комедија © 2026
    # Бул. Климент Охридски бр.27, Скопје 1000, Северна Македонија
    # Администрација: 02 310 9999
    # Билетарница: 071 330 233
    # E-mail: teatarkomedija@gmail.com
    clean_text_part = raw_text.replace("НУ Театар Комедија © 2026", "").strip()
    full_address = clean_text_part.split('Администрација')[0].strip().rstrip(', ')

    parts = full_address.split(',', 1)
    location = parts[0].strip()
    city = parts[1].strip() if len(parts) > 1 else "Скопје"

    return {"location": location, "city": city}

In [ ]:
def get_theater_description(url_link):
    try:
        details_page = requests.get(url_link)
        details = BeautifulSoup(details_page.text, 'html.parser')

        description_tag = details.select_one("div.qodef-grid-item")

        if description_tag:
            return clean_text(description_tag.get_text())
        return None

    except Exception as e:
        print(f"Error scraping {url_link}: {e}!")
        return None

In [ ]:
all_theater_show_data = []
seen_titles = set()

for show in all_cards[:]:

    title_tag = show.select_one("div.qodef-e-content > h5")
    title = clean_text(title_tag.get_text()) if title_tag else None
    if not title or title in seen_titles:
        continue

    image_tag = show.select_one("div.qodef-e-media-image > a > img")
    if image_tag and image_tag.get("src"):
        image_url = image_tag["src"]
    else:
        image_url = None

    link_tag = show.select_one("div.qodef-e-media-image > a")
    link = link_tag['href'] if link_tag and 'href' in link_tag.attrs else None


    description = get_theater_description(link) if link else ""

    show_entry = {
        "title": title,
        "image": image_url,
        "location": theater_address_info["location"],
        "city": theater_address_info["city"],
        "price": "600ден",
        "time_start": "20:00",
        "date_start": "",
        "description": description,
    }

    all_theater_show_data.append(show_entry)
    seen_titles.add(title)

# print(all_theater_show_data)
print(json.dumps(all_theater_show_data, indent=4, ensure_ascii=False))

In [ ]:
len(all_theater_show_data)

In [ ]:
df_theater_show_data = pd.DataFrame(data=all_theater_show_data)

df_theater_show_data[10:20]

In [ ]:
theater_show_titles = df_theater_show_data["title"].unique()

theater_show_titles

In [ ]:
len(theater_show_titles)

In [ ]:
# Тестирање за сите описи по театарска претстава.

for index, link in enumerate(
        unique_detail_links):
    try:
        details_page = requests.get(link)
        details = BeautifulSoup(details_page.text, 'html.parser')

        description_tag = details.select_one("div.qodef-grid-item")

        cleaned_description = clean_text(description_tag.get_text()) if description_tag else None

        show_description = {
            "description": cleaned_description
        }

        print(f"Show: {index}: {link}")
        print(show_description)

    except Exception as e:
        print(f"Error scraping {link}: {e}!")

In [ ]:
# Кодови за скрејпање на цена, време на почнување и датум на почнување.

In [ ]:
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# import time
#
#
# # =========================
# # 1. BROWSER SETUP (Brave)
# # =========================
# options = Options()
# options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
#
# options.add_argument("--start-maximized")
# options.add_argument("--disable-blink-features=AutomationControlled")
#
# driver = webdriver.Chrome(options=options)
# wait = WebDriverWait(driver, 20)
#
#
# # =========================
# # 2. OPEN PAGE
# # =========================
# driver.get("https://online.teatarkomedija.mk/")
# time.sleep(3)
#
#
# # =========================
# # 3. SCROLL (important for Elementor)
# # =========================
# driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
# time.sleep(2)
#
#
# # =========================
# # 4. CLICK BUTTON (ROBUST)
# # =========================
# try:
#     # try common Elementor buttons
#     buttons = driver.find_elements(By.CSS_SELECTOR, "a.elementor-button, button.elementor-button, .elementor-button")
#
#     print("Buttons found:", len(buttons))
#
#     clicked = False
#
#     for b in buttons:
#         if b.is_displayed():
#             driver.execute_script("arguments[0].scrollIntoView(true);", b)
#             time.sleep(1)
#             driver.execute_script("arguments[0].click();", b)
#             clicked = True
#             print("Button clicked")
#             break
#
#     if not clicked:
#         print("No clickable button found")
#
# except Exception as e:
#     print("Click error:", e)
#
#
# # =========================
# # 5. WAIT FOR EVENT DATA
# # =========================
# time.sleep(4)
#
#
# # =========================
# # 6. SCRAPE EVENTS
# # =========================
# events = []
#
# try:
#     cards = driver.find_elements(By.CSS_SELECTOR, ".card-body.eventInfoHeaderCardBody")
#
#     print("Events found:", len(cards))
#
#     for card in cards:
#         try:
#             name = card.find_element(By.ID, "eventName").text
#         except:
#             name = None
#
#         try:
#             director = card.find_element(By.ID, "eventDirector").text
#         except:
#             director = None
#
#         try:
#             place = card.find_element(By.CSS_SELECTOR, ".eventPlace").text
#         except:
#             place = None
#
#         try:
#             price = card.find_element(By.CSS_SELECTOR, ".eventPriceText").text
#         except:
#             price = None
#
#         event = {
#             "name": name,
#             "director": director,
#             "place": place,
#             "price": price
#         }
#
#         events.append(event)
#
#         print("\n--- EVENT ---")
#         print(event)
#
# except Exception as e:
#     print("Scraping error:", e)
#
#
# # =========================
# # 7. DEBUG (optional)
# # =========================
# driver.save_screenshot("debug.png")
#
#
# # =========================
# # 8. CLEAN EXIT
# # =========================
# driver.quit()
#
#
# # =========================
# # 9. RESULT
# # =========================
# print("\nTOTAL EVENTS:", len(events))

In [ ]:
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.common.by import By
# import time
#
#
# # =========================
# # 1. SETUP BRAVE
# # =========================
# options = Options()
# options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
# options.add_argument("--start-maximized")
#
# driver = webdriver.Chrome(options=options)
#
#
# # =========================
# # 2. OPEN PAGE
# # =========================
# driver.get("https://online.teatarkomedija.mk/")
# time.sleep(3)
#
#
# # =========================
# # 3. AUTO SCROLL (LOAD ALL EVENTS)
# # =========================
# last_height = driver.execute_script("return document.body.scrollHeight")
#
# while True:
#     driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
#     time.sleep(2)
#
#     new_height = driver.execute_script("return document.body.scrollHeight")
#
#     if new_height == last_height:
#         break
#     last_height = new_height
#
#
# # =========================
# # 4. SCRAPE ALL EVENTS
# # =========================
# cards = driver.find_elements(By.CSS_SELECTOR, ".card-body.eventInfoHeaderCardBody")
#
# events = []
#
# for card in cards:
#     try:
#         name = card.find_element(By.ID, "eventName").text
#     except:
#         name = None
#
#     try:
#         director = card.find_element(By.ID, "eventDirector").text
#     except:
#         director = None
#
#     try:
#         place = card.find_element(By.CSS_SELECTOR, ".eventPlace").text
#     except:
#         place = None
#
#     try:
#         price = card.find_element(By.CSS_SELECTOR, ".eventPriceText").text
#     except:
#         price = None
#
#     events.append({
#         "name": name,
#         "director": director,
#         "place": place,
#         "price": price
#     })
#
#
# # =========================
# # 5. RESULT
# # =========================
# for i, e in enumerate(events):
#     print(f"\n--- EVENT {i} ---")
#     print(e)
#
# print("\nTOTAL EVENTS:", len(events))
#
#
# driver.quit()

In [ ]:
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.common.by import By
# import time
#
#
# # =========================
# # 1. SETUP
# # =========================
# options = Options()
# options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
# options.add_argument("--start-maximized")
#
# driver = webdriver.Chrome(options=options)
#
#
# # =========================
# # 2. OPEN PAGE
# # =========================
# driver.get("https://online.teatarkomedija.mk/")
# time.sleep(5)
#
#
# # =========================
# # 3. FIND ALL CLICKABLE EVENT ITEMS
# # =========================
# events_data = []
#
# # ⚠️ ова мора да го прилагодиш ако има button/anchor per event
# event_buttons = driver.find_elements(By.CSS_SELECTOR, ".card-body.eventInfoHeaderCardBody, .card, a, button")
#
# print("Possible clickable items:", len(event_buttons))
#
#
# # =========================
# # 4. LOOP EACH ITEM (CLICK → SCRAPE → BACK)
# # =========================
# for i in range(len(event_buttons)):
#
#     # re-fetch each time (IMPORTANT for DOM refresh)
#     buttons = driver.find_elements(By.CSS_SELECTOR, ".card-body.eventInfoHeaderCardBody")
#
#     if i >= len(buttons):
#         break
#
#     try:
#         item = buttons[i]
#
#         driver.execute_script("arguments[0].scrollIntoView(true);", item)
#         time.sleep(1)
#
#         driver.execute_script("arguments[0].click();", item)
#         time.sleep(3)
#
#         # =========================
#         # 5. SCRAPE DETAIL PAGE
#         # =========================
#         name = driver.find_element(By.ID, "eventName").text
#         director = driver.find_element(By.ID, "eventDirector").text
#         place = driver.find_element(By.CSS_SELECTOR, ".eventPlace").text
#         price = driver.find_element(By.CSS_SELECTOR, ".eventPriceText").text
#
#         events_data.append({
#             "name": name,
#             "director": director,
#             "place": place,
#             "price": price
#         })
#
#         print(f"\n✔ Scraped event {i}: {name}")
#
#         # =========================
#         # 6. GO BACK
#         # =========================
#         driver.back()
#         time.sleep(4)
#
#     except Exception as e:
#         print(f"Error on item {i}:", e)
#         driver.get("https://online.teatarkomedija.mk/")
#         time.sleep(5)
#
#
# # =========================
# # 7. RESULT
# # =========================
# for i, e in enumerate(events_data):
#     print(f"\n--- EVENT {i} ---")
#     print(e)
#
# print("\nTOTAL EVENTS:", len(events_data))
#
#
# driver.quit()

In [ ]:
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.common.by import By
# import time
#
#
# options = Options()
# options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
# options.add_argument("--start-maximized")
#
# driver = webdriver.Chrome(options=options)
#
# driver.get("https://online.teatarkomedija.mk/")
# time.sleep(5)
#
# events = []
#
#
# # =========================
# # MAIN LOOP (CLICK STATE MACHINE)
# # =========================
# for i in range(100):  # safety limit
#
#     try:
#         # ALWAYS re-grab current active event card
#         card = driver.find_element(By.CSS_SELECTOR, ".card-body.eventInfoHeaderCardBody")
#
#         name = driver.find_element(By.ID, "eventName").text
#         director = driver.find_element(By.ID, "eventDirector").text
#         place = driver.find_element(By.CSS_SELECTOR, ".eventPlace").text
#         price = driver.find_element(By.CSS_SELECTOR, ".eventPriceText").text
#
#         events.append({
#             "name": name,
#             "director": director,
#             "place": place,
#             "price": price
#         })
#
#         print(f"✔ scraped: {name}")
#
#         # FIND NEXT ITEM (IMPORTANT)
#         # click next "card-title" or navigation item
#
#         next_buttons = driver.find_elements(By.CSS_SELECTOR, ".card-title, .eventTitle, .card")
#
#         if len(next_buttons) > 1:
#             driver.execute_script("arguments[0].click();", next_buttons[1])
#             time.sleep(2)
#         else:
#             break
#
#     except Exception as e:
#         print("Finished or error:", e)
#         break
#
#
# driver.quit()
#
# print("\nTOTAL EVENTS:", len(events))

In [ ]:
# import asyncio
# from playwright.async_api import async_playwright
#
# URL = "https://online.teatarkomedija.mk/"
#
# events = []
#
# async def main():
#
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=False)
#         page = await browser.new_page()
#
#         # =========================
#         # CAPTURE API RESPONSES
#         # =========================
#         async def handle_response(response):
#             global events
#
#             if "get_all_events.php" in response.url:
#                 try:
#                     data = await response.json()
#
#                     print("\n🔥 API CAPTURED")
#
#                     if isinstance(data, dict) and data.get("res") is False:
#                         print("API ERROR:", data.get("errorMessage"))
#                         return
#
#                     if isinstance(data, list):
#                         for item in data:
#                             if isinstance(item, dict):
#                                 events.append({
#                                     "id": item.get("id"),
#                                     "name": item.get("name"),
#                                     "place": item.get("place"),
#                                     "price": item.get("price")
#                                 })
#
#                     print("✔ events so far:", len(events))
#
#                 except Exception as e:
#                     print("parse error:", e)
#
#         page.on("response", lambda r: asyncio.create_task(handle_response(r)))
#
#         # =========================
#         # OPEN PAGE
#         # =========================
#         await page.goto(URL)
#
#         await page.wait_for_timeout(8000)
#
#         await page.reload()
#         await page.wait_for_timeout(8000)
#
#         await browser.close()
#
#
# # =========================
# # RUN IN JUPYTER
# # =========================
# await main()
#
#
# # =========================
# # OUTPUT
# # =========================
# print("\nTOTAL EVENTS:", len(events))
#
# for e in events:
#     print(e)